# PropIQ — Week 5
## Bronze → Silver Candidate Transformation

**Project:** P15 PropIQ — Real Estate Market Analytics
**Notebook path:** `notebooks/03_silver_transformations.ipynb`
**Team:** Ms. Thota Madhulika, Ms. P. Lakshmi Naga Sree, Ms. Vadlamuru Rishitha
**Technology:** Databricks Free Edition · Spark SQL · Delta tables

### The Week 5 task

Week 4 created four PropIQ Bronze Delta tables. Week 5 uses those tables to
create four **Silver Candidate** tables that are typed, standardised and
carry the calculated fields needed for Data Quality assessment in Week 6.

`Bronze tables → standardise existing columns → convert data types → calculate
documented fields → write Silver Candidate → reconcile counts and row-level lineage`

Silver Candidate is **row-preserving**. No row is filtered, deduplicated,
rejected or quarantined in this notebook — that is Week 6/7 work, driven by
the approved `dq_requirements.md` rules (P15-DQ-01 through P15-DQ-08).


## ⚠️ Before you run this notebook — confirm your Week-4 Bronze schema

The approved Data Pack (`data_dictionary.md`, v1.0 APPROVED) defines these
business columns per source:

| Source | Approved business columns |
|---|---|
| `listings.parquet` | `record_uid, listing_id, locality_id, broker_id, property_type, bedrooms, furnishing, built_up_area_sqft, asking_price_inr, price_per_sqft, listing_created_date, last_updated_timestamp, completion_date, listing_status, source_system, source_record_id, batch_id` |
| `leads.csv` | `record_uid, lead_id, listing_id, lead_timestamp, lead_channel, buyer_intent, qualified_flag, lead_status, budget_band, source_system, source_record_id, batch_id` |
| `localities.json` | `locality_id, locality_name, city, city_zone, market_segment, reference_price_per_sqft, active_flag, source_system, batch_id, record_uid` |
| `brokers.csv` | `broker_id, agency_name, city, broker_tier, active_flag, onboarded_date, service_rating, source_system, batch_id, record_uid` |

**If your `bronze_propiq_leads`, `bronze_propiq_localities` or
`bronze_propiq_brokers` tables were built with a narrower CSV/JSON
`CREATE TEMP VIEW ... (col STRING, ...)` schema than the list above, this
Silver notebook will fail with `column not found`.** Go back to
`02_bronze_ingestion.ipynb`, add the missing columns to each source view's
declared schema, rerun Part 4/5/6, and confirm with `DESCRIBE bronze_propiq_leads`
(etc.) that every column above is present before continuing here.


## 1. Outcome first — what must this notebook produce?

| Bronze input | Silver Candidate output | Main work |
|---|---|---|
| `bronze_propiq_listings` | `silver_propiq_listings_candidate` | standardise identifiers/domains, type nine fields, calculate six new fields |
| `bronze_propiq_leads` | `silver_propiq_leads_candidate` | standardise identifiers/domains, type `lead_timestamp` and `qualified_flag` |
| `bronze_propiq_localities` | `silver_propiq_localities_candidate` | standardise identifiers/domains, type `reference_price_per_sqft` and `active_flag` |
| `bronze_propiq_brokers` | `silver_propiq_brokers_candidate` | standardise identifiers/domains, type `service_rating`, `onboarded_date` and `active_flag` |

### Six new listings columns

| New column | Simple meaning | Formula | Supports |
|---|---|---|---|
| `actual_days_on_market` | days the listing was open, only when it has a completion date | `datediff(completion_date, listing_created_date)` when `completion_date IS NOT NULL` | DQ-05 |
| `days_since_last_update` | staleness of the source record | `datediff(current_date(), last_updated_timestamp)` | data freshness |
| `is_completed` | whether the listing reached a finished state | `listing_status IN ('sold','rented')` | DQ-05 |
| `is_chronology_valid` | whether the completion date makes sense against the creation date | `completion_date IS NULL OR completion_date >= listing_created_date` | DQ-05 |
| `calculated_price_per_sqft` | price per square foot recomputed from typed inputs | `ROUND(asking_price_inr / built_up_area_sqft)` when area `> 0` | DQ-04 |
| `price_per_sqft_variance` | gap between the stored and recalculated price per sq ft | `price_per_sqft - calculated_price_per_sqft` | DQ-04 |

Every calculated field uses **typed** inputs from Stage 5.2, never the raw
Bronze string/columns directly, and every field stays `NULL` when an input is
missing or unparseable rather than being guessed.


## 2. How to use this notebook

This is an executable Databricks lab. Use the same cycle throughout:

1. **Understand** the single purpose stated above the cell.
2. **Run** only that cell.
3. **Inspect** the result described below it.
4. **Resolve** any mismatch before moving ahead.

The first attempt should be step by step; do not start with **Run all**.

### Week 4 handoff and Week 5 boundary

### Prerequisites

- Week 4 is complete and corrected per the callout above.
- The four PropIQ Bronze tables exist with the full approved column set.
- You know the catalog and schema used in Week 4 (`workspace.default`).

This notebook reads the completed Bronze tables and writes new Candidate
tables. It does not touch or recreate Bronze. It does not filter, deduplicate,
join across entities, quarantine or produce Gold metrics — those are Week
6/7/8 work.


## 3. Confirm the Week-4 handoff

### 3.1 Select the working location

**Purpose:** use the same catalog and schema that contain the completed Week-4 Bronze tables.


In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

SELECT current_catalog() AS active_catalog,
       current_schema()  AS active_schema;


**Expected result:** one row showing `workspace` and `default`. If your team used another approved location, change only these two settings.


### 3.2 Confirm the Bronze inputs

**Purpose:** verify that all four required PropIQ Bronze tables exist before writing any Week-5 output.


In [0]:
%sql
SHOW TABLES LIKE 'bronze_propiq_*';


database,tableName,isTemporary
default,bronze_propiq_brokers,false
default,bronze_propiq_leads,false
default,bronze_propiq_listings,false
default,bronze_propiq_localities,false


**Expected result:** `bronze_propiq_listings`, `bronze_propiq_leads`, `bronze_propiq_localities` and `bronze_propiq_brokers` are listed. **Stop here** if any table is missing; repair Week 4 first.


### 3.3 Confirm the full approved column set is present

**Purpose:** catch the narrower Week-4 schema problem described in the callout above before it causes a `column not found` error mid-notebook.


In [0]:
%sql
DESCRIBE bronze_propiq_listings;


col_name,data_type,comment
record_uid,string,null
listing_id,string,null
locality_id,string,null
broker_id,string,null
property_type,string,null
bedrooms,bigint,null
furnishing,string,null
built_up_area_sqft,bigint,null
asking_price_inr,bigint,null
price_per_sqft,bigint,null


In [0]:
%sql
DESCRIBE bronze_propiq_leads;


col_name,data_type,comment
record_uid,string,null
lead_id,string,null
listing_id,string,null
lead_channel,string,null
buyer_intent,string,null
qualified_flag,string,null
lead_status,string,null
_source_file_name,string,null
_source_file_path,string,null
_ingested_at,timestamp,null


In [0]:
%sql
DESCRIBE bronze_propiq_localities;


col_name,data_type,comment
locality_id,string,null
locality_name,string,null
city,string,null
city_zone,string,null
market_segment,string,null
_source_file_name,string,null
_source_file_path,string,null
_ingested_at,timestamp,null
_ingestion_run_id,string,null
_schema_version,string,null


In [0]:
%sql
DESCRIBE bronze_propiq_brokers;


col_name,data_type,comment
broker_id,string,null
agency_name,string,null
broker_tier,string,null
service_rating,string,null
_source_file_name,string,null
_source_file_path,string,null
_ingested_at,timestamp,null
_ingestion_run_id,string,null
_schema_version,string,null
_rescued_payload,string,null


**Expected result:** each `DESCRIBE` output includes every business column listed in the Section-0 table for that source, plus the technical `_source_file_name`, `_source_file_path`, `_ingested_at`, `_ingestion_run_id`, `_schema_version`, `_record_hash` columns (`_rescued_payload` too for CSV/JSON sources). If any business column is missing, stop and correct Week 4 before continuing.


### 3.4 Record the starting counts

**Purpose:** capture the Bronze row count for each entity. These counts become the baseline for final reconciliation.


In [0]:
%sql
SELECT 'listings' AS entity, COUNT(*) AS bronze_rows FROM bronze_propiq_listings
UNION ALL
SELECT 'leads', COUNT(*) FROM bronze_propiq_leads
UNION ALL
SELECT 'localities', COUNT(*) FROM bronze_propiq_localities
UNION ALL
SELECT 'brokers', COUNT(*) FROM bronze_propiq_brokers;


entity,bronze_rows
listings,50200
leads,120800
localities,80
brokers,320


**Expected result:** one count for each entity. Per `source_manifest.csv` the approved counts are 50,200 listings, 120,800 leads, 80 localities and 320 brokers — this reference does not invent them, it only tells you what to expect from the approved Data Pack.


## 4. Understand the transformation patterns

Before building a complete table, learn the three SQL patterns used throughout this notebook.

### Pattern A — standardise a controlled field

`TRIM` removes spaces at the beginning/end. `UPPER` gives identifiers one
representation; `INITCAP`/`LOWER` gives a controlled domain field one
representation, matching the casing used in `data_dictionary.md`.


In [0]:
%sql
SELECT listing_id                     AS bronze_listing_id,
       upper(trim(listing_id))        AS candidate_listing_id,
       listing_status                 AS bronze_listing_status,
       lower(trim(listing_status))    AS candidate_listing_status,
       property_type                  AS bronze_property_type,
       initcap(trim(property_type))   AS candidate_property_type
FROM bronze_propiq_listings
LIMIT 10;


bronze_listing_id,candidate_listing_id,bronze_listing_status,candidate_listing_status,bronze_property_type,candidate_property_type
LST-0000001,LST-0000001,active,active,Apartment,Apartment
LST-0000002,LST-0000002,active,active,Apartment,Apartment
LST-0000003,LST-0000003,rented,rented,Apartment,Apartment
LST-0000004,LST-0000004,paused,paused,Apartment,Apartment
LST-0000005,LST-0000005,sold,sold,Apartment,Apartment
LST-0000006,LST-0000006,sold,sold,Apartment,Apartment
LST-0000007,LST-0000007,expired,expired,Villa,Villa
LST-0000008,LST-0000008,active,active,Apartment,Apartment
LST-0000009,LST-0000009,rented,rented,Apartment,Apartment
LST-0000010,LST-0000010,sold,sold,Villa,Villa


**Expected result:** Bronze and Candidate representations appear side by side. Do not apply casing to free-text or synthetic label fields (e.g. `agency_name`, `locality_name`) beyond `TRIM` + `INITCAP`, and never invent a new category that is not already present in Bronze.


### Pattern B — convert a type safely

`TRY_CAST` converts valid values and returns `NULL` for an unparseable value. The physical row remains present.


In [0]:
%sql
SELECT listing_created_date                    AS bronze_listing_created_date,
       try_cast(listing_created_date AS DATE)  AS candidate_listing_created_date,
       asking_price_inr                        AS bronze_asking_price_inr,
       try_cast(asking_price_inr AS BIGINT)     AS candidate_asking_price_inr
FROM bronze_propiq_listings
LIMIT 10;


bronze_listing_created_date,candidate_listing_created_date,bronze_asking_price_inr,candidate_asking_price_inr
2025-05-15,2025-05-15,11005988,11005988
2025-07-17,2025-07-17,21793252,21793252
2024-12-23,2024-12-23,16873096,16873096
2024-10-31,2024-10-31,21144060,21144060
2025-07-28,2025-07-28,9571528,9571528
2024-05-25,2024-05-25,10529001,10529001
2025-10-02,2025-10-02,34199100,34199100
2024-01-30,2024-01-30,12205728,12205728
2025-02-04,2025-02-04,9395822,9395822
2024-02-12,2024-02-12,39615810,39615810


**Expected result:** successfully parsed values have proper types. An invalid non-null Bronze value may become `NULL`; Week 5 records that outcome but does not remove the row.


### Pattern C — create a documented calculated field

First type the input columns. Then calculate from those typed values. Never invent a formula from a column name.


In [0]:
%sql
WITH example AS (
  SELECT try_cast(asking_price_inr AS BIGINT)      AS asking_price_inr,
         try_cast(built_up_area_sqft AS INT)        AS built_up_area_sqft,
         try_cast(price_per_sqft AS BIGINT)         AS price_per_sqft
  FROM bronze_propiq_listings
)
SELECT asking_price_inr,
       built_up_area_sqft,
       price_per_sqft,
       CASE WHEN built_up_area_sqft > 0
            THEN round(asking_price_inr / built_up_area_sqft)
       END AS calculated_price_per_sqft
FROM example
LIMIT 10;


asking_price_inr,built_up_area_sqft,price_per_sqft,calculated_price_per_sqft
11005988,1631,6748,6748.0
21793252,1157,18836,18836.0
16873096,1973,8552,8552.0
21144060,1164,18165,18165.0
9571528,1478,6476,6476.0
10529001,1827,5763,5763.0
34199100,4740,7215,7215.0
12205728,1296,9418,9418.0
9395822,1301,7222,7222.0
39615810,3882,10205,10205.0


**Expected result:** the inputs and calculated value are visible together. If either input cannot be typed, or the area is zero, the calculated value remains `NULL` rather than being guessed.


## 5. Build listings Candidate — complete worked example

Listings is the full teaching example and the entity most Week-6 DQ rules
depend on (DQ-01 through DQ-05). Build it in five small stages:

1. standardise identifiers, domains and lineage pass-through;
2. convert dates, counts, area and price fields;
3. inspect the typed result;
4. calculate market-duration and chronology flags;
5. calculate and compare price per square foot, then write the Delta table.

### 5.1 Standardise identifiers and domain fields

**Purpose:** create consistent representations for controlled identifiers and
approved domain values. No dates, counts or prices are converted in this cell.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW listings_standardised AS
SELECT
  upper(trim(record_uid))        AS record_uid,
  upper(trim(listing_id))        AS listing_id,
  upper(trim(locality_id))       AS locality_id,
  upper(trim(broker_id))         AS broker_id,
  initcap(trim(property_type))   AS property_type,
  bedrooms,
  initcap(trim(furnishing))      AS furnishing,
  built_up_area_sqft,
  asking_price_inr,
  price_per_sqft,
  listing_created_date,
  last_updated_timestamp,
  completion_date,
  lower(trim(listing_status))    AS listing_status,
  upper(trim(source_system))     AS source_system,
  upper(trim(source_record_id))  AS source_record_id,
  upper(trim(batch_id))          AS batch_id,
  _source_file_name, _source_file_path, _ingested_at,
  _ingestion_run_id, _schema_version, _record_hash
FROM bronze_propiq_listings;


**Expected result:** `listings_standardised` is created. IDs use one case, `listing_status` uses one case, `property_type`/`furnishing` use one case, and every other Bronze value (including counts, prices and dates that are converted in the next stage) is still present.


### 5.2 Convert the documented data types

**Purpose:** convert counts, area, price and date/timestamp fields safely. `TRY_CAST` keeps the row even when a value cannot be parsed.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW listings_typed AS
SELECT
  record_uid, listing_id, locality_id, broker_id, property_type,
  try_cast(bedrooms AS INT) AS bedrooms,
  furnishing,
  try_cast(built_up_area_sqft AS INT) AS built_up_area_sqft,
  try_cast(asking_price_inr AS BIGINT) AS asking_price_inr,
  try_cast(price_per_sqft AS BIGINT) AS price_per_sqft,
  try_cast(listing_created_date AS DATE) AS listing_created_date,
  try_cast(last_updated_timestamp AS TIMESTAMP) AS last_updated_timestamp,
  try_cast(completion_date AS DATE) AS completion_date,
  listing_status, source_system, source_record_id, batch_id,
  _source_file_name, _source_file_path, _ingested_at,
  _ingestion_run_id, _schema_version, _record_hash
FROM listings_standardised;


**Expected result:** `listings_typed` is created. Bedrooms and area are `INT`, prices are `BIGINT`, dates are `DATE`, the update timestamp is `TIMESTAMP`, and lineage remains available.


### 5.3 Inspect the typed result

**Purpose:** verify the schema before using typed fields in calculations.


In [0]:
%sql
DESCRIBE listings_typed;


col_name,data_type,comment
record_uid,string,null
listing_id,string,null
locality_id,string,null
broker_id,string,null
property_type,string,null
bedrooms,int,null
furnishing,string,null
built_up_area_sqft,int,null
asking_price_inr,bigint,null
price_per_sqft,bigint,null


**Expected result:** `bedrooms`/`built_up_area_sqft` are `INT`, `asking_price_inr`/`price_per_sqft` are `BIGINT`, `listing_created_date`/`completion_date` are `DATE`, `last_updated_timestamp` is `TIMESTAMP`, and the technical fields remain present.


### 5.4 Calculate market-duration and chronology flags

**Purpose:** add four documented fields using the typed dates and standardised status. These flags directly support DQ-05 (completion status and chronology) in Week 6.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW listings_with_time_measures AS
SELECT
  *,
  CASE WHEN completion_date IS NOT NULL AND listing_created_date IS NOT NULL
       THEN datediff(completion_date, listing_created_date) END AS actual_days_on_market,
  CASE WHEN last_updated_timestamp IS NOT NULL
       THEN datediff(current_date(), to_date(last_updated_timestamp)) END AS days_since_last_update,
  CASE WHEN listing_status IS NULL THEN NULL
       ELSE listing_status IN ('sold', 'rented') END AS is_completed,
  CASE WHEN completion_date IS NULL THEN NULL
       ELSE completion_date >= listing_created_date END AS is_chronology_valid
FROM listings_typed;


**Expected result:** the view has four new fields. If a required input is null or unparseable, the related calculated value is null rather than guessed. `is_chronology_valid = false` is evidence for DQ-05, not a reason to drop the row here.


### 5.5 Calculate and compare price per square foot

**Purpose:** calculate `price_per_sqft` from its typed components and compare it with the stored `price_per_sqft` value. This directly supports DQ-04 (price-per-square-foot reconciliation).


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW listings_candidate_ready AS
SELECT
  *,
  CASE WHEN asking_price_inr IS NOT NULL AND built_up_area_sqft IS NOT NULL AND built_up_area_sqft > 0
       THEN cast(round(asking_price_inr / built_up_area_sqft) AS BIGINT) END AS calculated_price_per_sqft,
  CASE WHEN price_per_sqft IS NOT NULL AND asking_price_inr IS NOT NULL
            AND built_up_area_sqft IS NOT NULL AND built_up_area_sqft > 0
       THEN price_per_sqft - cast(round(asking_price_inr / built_up_area_sqft) AS BIGINT)
       END AS price_per_sqft_variance,
  current_timestamp() AS _candidate_created_at,
  'propiq_silver_candidate_v1.0' AS _candidate_schema_version
FROM listings_with_time_measures;


**Expected result:** the ready view contains all six new business fields plus Candidate creation time and schema version. `price_per_sqft_variance = 0` means the stored and calculated price per sq ft agree; any other value is retained for DQ-04 in Week 6, not corrected here.


### 5.6 Inspect the calculations

**Purpose:** view the typed inputs beside the six calculated outputs before writing the table.


In [0]:
%sql
SELECT listing_id, listing_created_date, completion_date, last_updated_timestamp, listing_status,
       asking_price_inr, built_up_area_sqft, price_per_sqft,
       actual_days_on_market, days_since_last_update, is_completed, is_chronology_valid,
       calculated_price_per_sqft, price_per_sqft_variance
FROM listings_candidate_ready
LIMIT 10;


listing_id,listing_created_date,completion_date,last_updated_timestamp,listing_status,asking_price_inr,built_up_area_sqft,price_per_sqft,actual_days_on_market,days_since_last_update,is_completed,is_chronology_valid,calculated_price_per_sqft,price_per_sqft_variance
LST-0000001,2025-05-15,null,2025-09-03T21:21:36.000Z,active,11005988,1631,6748,null,338,false,null,6748,0
LST-0000002,2025-07-17,null,2025-08-10T14:05:46.000Z,active,21793252,1157,18836,null,362,false,null,18836,0
LST-0000003,2024-12-23,2025-03-01,2025-01-02T13:35:06.000Z,rented,16873096,1973,8552,68,582,true,true,8552,0
LST-0000004,2024-10-31,null,2024-11-08T16:45:22.000Z,paused,21144060,1164,18165,null,637,false,null,18165,0
LST-0000005,2025-07-28,2025-12-04,2025-10-17T21:50:45.000Z,sold,9571528,1478,6476,129,294,true,true,6476,0
LST-0000006,2024-05-25,2024-09-26,2024-08-28T03:01:36.000Z,sold,10529001,1827,5763,124,709,true,true,5763,0
LST-0000007,2025-10-02,null,2025-11-22T06:51:33.000Z,expired,34199100,4740,7215,null,258,false,null,7215,0
LST-0000008,2024-01-30,null,2024-05-18T02:03:22.000Z,active,12205728,1296,9418,null,811,false,null,9418,0
LST-0000009,2025-02-04,2025-06-09,2025-06-11T17:21:51.000Z,rented,9395822,1301,7222,125,422,true,true,7222,0
LST-0000010,2024-02-12,2024-05-01,2024-04-30T07:39:38.000Z,sold,39615810,3882,10205,79,829,true,true,10205,0


**Expected result:** each calculated value can be explained directly from the typed inputs shown in the same row.


Check only genuine parse failures: a non-null Bronze value that became null after conversion.


In [0]:
%sql
SELECT
  SUM(CASE WHEN listing_created_date IS NOT NULL AND try_cast(listing_created_date AS DATE) IS NULL THEN 1 ELSE 0 END) AS created_date_parse_failures,
  SUM(CASE WHEN completion_date IS NOT NULL AND try_cast(completion_date AS DATE) IS NULL THEN 1 ELSE 0 END) AS completion_date_parse_failures,
  SUM(CASE WHEN built_up_area_sqft IS NOT NULL AND try_cast(built_up_area_sqft AS INT) IS NULL THEN 1 ELSE 0 END) AS area_parse_failures,
  SUM(CASE WHEN asking_price_inr IS NOT NULL AND try_cast(asking_price_inr AS BIGINT) IS NULL THEN 1 ELSE 0 END) AS price_parse_failures
FROM bronze_propiq_listings;


created_date_parse_failures,completion_date_parse_failures,area_parse_failures,price_parse_failures
0,100,0,0


**Expected result:** actual failure counts from your data. A non-zero value is evidence for Week 6 — not permission to delete or repair the row in Week 5.


### 5.7 Write the listings Candidate table

**Purpose:** persist the completed listings result as a Delta table.


In [0]:
%sql
CREATE OR REPLACE TABLE silver_propiq_listings_candidate
USING DELTA
AS SELECT * FROM listings_candidate_ready;


num_affected_rows,num_inserted_rows


**Expected result:** the listings Candidate Delta table is created. `CREATE OR REPLACE` supports the controlled snapshot rerun used later.


## 6. Build leads Candidate — guided practice

Leads needs standardisation plus two safe type conversions
(`lead_timestamp`, `qualified_flag`). It does not receive invented calculated
fields — chronology against listings (DQ-06) is a Week-6/7 join-and-DQ task,
not a Week-5 single-table calculation.


In [0]:
%sql
CREATE OR REPLACE TABLE silver_propiq_leads_candidate
USING DELTA
AS
SELECT
  upper(trim(record_uid))         AS record_uid,
  upper(trim(lead_id))            AS lead_id,
  upper(trim(listing_id))         AS listing_id,
  try_cast(lead_timestamp AS TIMESTAMP) AS lead_timestamp,
  initcap(trim(lead_channel))     AS lead_channel,
  initcap(trim(buyer_intent))     AS buyer_intent,
  try_cast(qualified_flag AS BOOLEAN) AS qualified_flag,
  lower(trim(lead_status))        AS lead_status,
  trim(budget_band)               AS budget_band,
  upper(trim(source_system))      AS source_system,
  upper(trim(source_record_id))   AS source_record_id,
  upper(trim(batch_id))           AS batch_id,
  _source_file_name, _source_file_path, _ingested_at,
  _ingestion_run_id, _schema_version AS _bronze_schema_version,
  _record_hash AS _bronze_record_hash, _rescued_payload,
  current_timestamp() AS _candidate_created_at,
  'propiq_silver_candidate_v1.0' AS _candidate_schema_version
FROM bronze_propiq_leads;


**Expected result:** a row-preserving leads Candidate table with standardised identifiers/domains, typed `lead_timestamp` and `qualified_flag`, and complete Bronze lineage.


Inspect only the key fields and lineage; avoid `SELECT *` when learning.


In [0]:
%sql
SELECT lead_id, listing_id, lead_timestamp, lead_channel, buyer_intent, qualified_flag, lead_status,
       _source_file_name, _bronze_record_hash
FROM silver_propiq_leads_candidate
LIMIT 10;


Check whether any non-null Bronze `lead_timestamp` or `qualified_flag` value failed conversion.


In [0]:
%sql
SELECT
  SUM(CASE WHEN lead_timestamp IS NOT NULL AND try_cast(lead_timestamp AS TIMESTAMP) IS NULL THEN 1 ELSE 0 END) AS lead_timestamp_parse_failures,
  SUM(CASE WHEN qualified_flag IS NOT NULL AND try_cast(qualified_flag AS BOOLEAN) IS NULL THEN 1 ELSE 0 END) AS qualified_flag_parse_failures
FROM bronze_propiq_leads;


**Expected result:** actual failure counts. Keep any affected row; Week 6 decides its DQ outcome under DQ-06/DQ-07.

**Checkpoint:** explain why each transformation above is allowed under the approved Week-5 specification — standardisation and safe type conversion only, nothing that changes row count or infers a business value.


## 7. Build localities Candidate — guided practice

Localities is reference-master data. It needs standardisation plus two safe
type conversions (`reference_price_per_sqft`, `active_flag`).


In [0]:
%sql
CREATE OR REPLACE TABLE silver_propiq_localities_candidate
USING DELTA
AS
SELECT
  upper(trim(locality_id))        AS locality_id,
  initcap(trim(locality_name))    AS locality_name,
  initcap(trim(city))             AS city,
  initcap(trim(city_zone))        AS city_zone,
  initcap(trim(market_segment))   AS market_segment,
  try_cast(reference_price_per_sqft AS BIGINT) AS reference_price_per_sqft,
  try_cast(active_flag AS BOOLEAN) AS active_flag,
  upper(trim(source_system))      AS source_system,
  upper(trim(batch_id))           AS batch_id,
  upper(trim(record_uid))         AS record_uid,
  _source_file_name, _source_file_path, _ingested_at,
  _ingestion_run_id, _schema_version AS _bronze_schema_version,
  _record_hash AS _bronze_record_hash, _rescued_payload,
  current_timestamp() AS _candidate_created_at,
  'propiq_silver_candidate_v1.0' AS _candidate_schema_version
FROM bronze_propiq_localities;


**Expected result:** a row-preserving localities Candidate table with standardised text fields, `reference_price_per_sqft` as `BIGINT`, `active_flag` as `BOOLEAN`, and complete Bronze lineage.


In [0]:
%sql
SELECT locality_id, locality_name, city, city_zone, market_segment,
       reference_price_per_sqft, active_flag,
       _source_file_name, _bronze_record_hash
FROM silver_propiq_localities_candidate
LIMIT 10;


**Checkpoint:** localities is a small master table (80 rows per the manifest). Use it to sanity-check `city`/`city_zone`/`market_segment` values against the approved domains before trusting the same pattern on the larger listings/leads tables.


## 8. Build brokers Candidate — guided practice

Brokers is reference-master data. It adds three type conversions
(`service_rating`, `onboarded_date`, `active_flag`) in addition to controlled
string standardisation, mirroring the single-clear-conversion pattern used
for master tables.


In [0]:
%sql
CREATE OR REPLACE TABLE silver_propiq_brokers_candidate
USING DELTA
AS
SELECT
  upper(trim(broker_id))          AS broker_id,
  initcap(trim(agency_name))      AS agency_name,
  initcap(trim(city))             AS city,
  initcap(trim(broker_tier))      AS broker_tier,
  try_cast(active_flag AS BOOLEAN) AS active_flag,
  try_cast(onboarded_date AS DATE) AS onboarded_date,
  try_cast(service_rating AS DECIMAL(3,1)) AS service_rating,
  upper(trim(source_system))      AS source_system,
  upper(trim(batch_id))           AS batch_id,
  upper(trim(record_uid))         AS record_uid,
  _source_file_name, _source_file_path, _ingested_at,
  _ingestion_run_id, _schema_version AS _bronze_schema_version,
  _record_hash AS _bronze_record_hash, _rescued_payload,
  current_timestamp() AS _candidate_created_at,
  'propiq_silver_candidate_v1.0' AS _candidate_schema_version
FROM bronze_propiq_brokers;


**Expected result:** a brokers Candidate Delta table with `service_rating` as `DECIMAL(3,1)`, `onboarded_date` as `DATE`, `active_flag` as `BOOLEAN`, and all rows preserved.


Check whether any non-null Bronze `service_rating` value failed conversion.


In [0]:
%sql
SELECT COUNT(*) AS service_rating_parse_failures
FROM bronze_propiq_brokers
WHERE service_rating IS NOT NULL
  AND try_cast(service_rating AS DECIMAL(3,1)) IS NULL;


**Expected result:** an actual failure count. Keep any affected row; Week 6 decides its DQ outcome.


## 9. Validate the complete Week-5 output

Validation answers two questions:

1. Did every Bronze physical row reach Candidate exactly once?
2. Can every Candidate row be traced back to Bronze?

### 9.1 Record reconciliation


In [0]:
%sql
WITH counts AS (
  SELECT 'listings' AS entity, (SELECT COUNT(*) FROM bronze_propiq_listings) AS bronze_rows,
         (SELECT COUNT(*) FROM silver_propiq_listings_candidate) AS candidate_rows
  UNION ALL
  SELECT 'leads', (SELECT COUNT(*) FROM bronze_propiq_leads),
         (SELECT COUNT(*) FROM silver_propiq_leads_candidate)
  UNION ALL
  SELECT 'localities', (SELECT COUNT(*) FROM bronze_propiq_localities),
         (SELECT COUNT(*) FROM silver_propiq_localities_candidate)
  UNION ALL
  SELECT 'brokers', (SELECT COUNT(*) FROM bronze_propiq_brokers),
         (SELECT COUNT(*) FROM silver_propiq_brokers_candidate)
)
SELECT *, candidate_rows - bronze_rows AS difference,
       CASE WHEN candidate_rows = bronze_rows THEN 'PASS' ELSE 'CHECK' END AS status
FROM counts;


**Expected result:** `difference = 0` and `status = PASS` for all four entities. A mismatch usually means a filter, `DISTINCT`, deduplication or unsafe join changed the grain.


### 9.2 Prove row-level lineage

A count match alone is not enough. Compare the Bronze record hashes with the
hashes retained in each Candidate table. Start with listings.


In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM (
     SELECT _record_hash FROM bronze_propiq_listings
     EXCEPT ALL
     SELECT _record_hash FROM silver_propiq_listings_candidate
  )) AS bronze_rows_missing_in_candidate,
  (SELECT COUNT(*) FROM (
     SELECT _record_hash FROM silver_propiq_listings_candidate
     EXCEPT ALL
     SELECT _record_hash FROM bronze_propiq_listings
  )) AS unexpected_candidate_rows;


**Expected result:** both listings values are zero. This proves that the same physical Bronze rows reached Candidate. (Listings keeps the column named `_record_hash` because Part 5 carried it through unrenamed; leads/localities/brokers renamed it to `_bronze_record_hash` — used in the next three checks.)


Run the same row-level proof for leads.


In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM (
     SELECT _record_hash FROM bronze_propiq_leads
     EXCEPT ALL
     SELECT _bronze_record_hash FROM silver_propiq_leads_candidate
  )) AS bronze_rows_missing_in_candidate,
  (SELECT COUNT(*) FROM (
     SELECT _bronze_record_hash FROM silver_propiq_leads_candidate
     EXCEPT ALL
     SELECT _record_hash FROM bronze_propiq_leads
  )) AS unexpected_candidate_rows;


**Expected result:** both leads values are zero.


Run the same row-level proof for localities.


In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM (
     SELECT _record_hash FROM bronze_propiq_localities
     EXCEPT ALL
     SELECT _bronze_record_hash FROM silver_propiq_localities_candidate
  )) AS bronze_rows_missing_in_candidate,
  (SELECT COUNT(*) FROM (
     SELECT _bronze_record_hash FROM silver_propiq_localities_candidate
     EXCEPT ALL
     SELECT _record_hash FROM bronze_propiq_localities
  )) AS unexpected_candidate_rows;


**Expected result:** both localities values are zero.


Run the same row-level proof for brokers.


In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM (
     SELECT _record_hash FROM bronze_propiq_brokers
     EXCEPT ALL
     SELECT _bronze_record_hash FROM silver_propiq_brokers_candidate
  )) AS bronze_rows_missing_in_candidate,
  (SELECT COUNT(*) FROM (
     SELECT _bronze_record_hash FROM silver_propiq_brokers_candidate
     EXCEPT ALL
     SELECT _record_hash FROM bronze_propiq_brokers
  )) AS unexpected_candidate_rows;


**Expected result:** both brokers values are zero. All four entities now have row-level lineage proof.


### 9.3 Confirm the output format


In [0]:
%sql
SELECT table_name, data_source_format
FROM system.information_schema.tables
WHERE table_catalog = current_catalog()
  AND table_schema = current_schema()
  AND table_name LIKE 'silver_propiq_%_candidate'
ORDER BY table_name;


**Expected result:** all four Candidate tables are listed with Delta as the storage format. If this system view is unavailable in your edition, use `DESCRIBE DETAIL <table_name>` per table.


### 9.4 Also confirm the required reconciliation identity holds at the physical-key level

**Purpose:** the Data Pack's reconciliation rule (`README.md`) is stated in
terms of `COUNT(DISTINCT record_uid)`, not raw row count. Prove the two
match for this notebook's scope — Silver Candidate distinct physical records
equal Bronze distinct physical records, since no dedup happens yet.


In [0]:
%sql
SELECT 'listings' AS entity,
       (SELECT COUNT(DISTINCT record_uid) FROM bronze_propiq_listings) AS bronze_distinct_uids,
       (SELECT COUNT(DISTINCT record_uid) FROM silver_propiq_listings_candidate) AS candidate_distinct_uids
UNION ALL
SELECT 'leads',
       (SELECT COUNT(DISTINCT record_uid) FROM bronze_propiq_leads),
       (SELECT COUNT(DISTINCT record_uid) FROM silver_propiq_leads_candidate)
UNION ALL
SELECT 'localities',
       (SELECT COUNT(DISTINCT record_uid) FROM bronze_propiq_localities),
       (SELECT COUNT(DISTINCT record_uid) FROM silver_propiq_localities_candidate)
UNION ALL
SELECT 'brokers',
       (SELECT COUNT(DISTINCT record_uid) FROM bronze_propiq_brokers),
       (SELECT COUNT(DISTINCT record_uid) FROM silver_propiq_brokers_candidate);


**Expected result:** the two counts match for every entity. Keep this evidence — Week 6/7 will extend this same identity to `Trusted Silver + Quarantine`.


## 10. Controlled repeat-run test

1. Record the current Candidate counts (Section 9.1).
2. Rerun the four `CREATE OR REPLACE TABLE` cells (Sections 5.7, 6, 7, 8).
3. Run the reconciliation cell (Section 9.1) again.

**Expected result:** counts remain stable. The table history may show a new write, but the same snapshot must not be appended again.

> Do not claim repeat-run safety merely because the SQL completed. The proof is stable counts plus reconciliation.


In [0]:
%sql
DESCRIBE HISTORY silver_propiq_listings_candidate;


**Expected result:** at least two `CREATE OR REPLACE TABLE AS SELECT` operations (the original build and the rerun), each with its own version number and timestamp.


## 11. Field-level transformation contract (viva / evidence reference)

| Entity | Standardisation | Type conversions | Calculated fields | DQ rules supported |
|---|---|---|---|---|
| listings | `UPPER(TRIM())` on IDs/keys; `INITCAP` on `property_type`/`furnishing`; `LOWER` on `listing_status` | `bedrooms`, `built_up_area_sqft` → `INT`; `asking_price_inr`, `price_per_sqft` → `BIGINT`; `listing_created_date`, `completion_date` → `DATE`; `last_updated_timestamp` → `TIMESTAMP` | `actual_days_on_market`, `days_since_last_update`, `is_completed`, `is_chronology_valid`, `calculated_price_per_sqft`, `price_per_sqft_variance` | DQ-01, DQ-02 (keys carried forward), DQ-03, DQ-04, DQ-05 |
| leads | `UPPER(TRIM())` on IDs/keys; `INITCAP` on `lead_channel`/`buyer_intent`; `LOWER` on `lead_status`; `TRIM` on `budget_band` | `lead_timestamp` → `TIMESTAMP`; `qualified_flag` → `BOOLEAN` | none | DQ-06, DQ-07, DQ-08 |
| localities | `UPPER(TRIM())` on IDs/keys; `INITCAP` on `locality_name`/`city`/`city_zone`/`market_segment` | `reference_price_per_sqft` → `BIGINT`; `active_flag` → `BOOLEAN` | none | DQ-02 |
| brokers | `UPPER(TRIM())` on IDs/keys; `INITCAP` on `agency_name`/`city`/`broker_tier` | `active_flag` → `BOOLEAN`; `onboarded_date` → `DATE`; `service_rating` → `DECIMAL(3,1)` | none | DQ-02 |

No entity was filtered, joined across sources, deduplicated, or had a value
guessed to replace a parse failure. Any non-zero parse-failure count recorded
in Sections 5–8 is carried into Week 6 as DQ evidence, not resolved here.


## 12. Common failures and recovery

| Problem | What to do |
|---|---|
| `bronze_propiq_leads`/`localities`/`brokers` column not found | your Week-4 `CREATE OR REPLACE TEMP VIEW ... USING CSV/JSON` schema is narrower than `data_dictionary.md` — add the missing columns and rerun Week 4 before continuing |
| Bronze table not found | select the correct Week-4 catalog/schema; do not recreate Bronze here |
| typed value becomes null | compare the non-null Bronze value and target type; retain the row |
| Candidate count changes | remove unintended filter, `DISTINCT`, deduplication or grain-changing join |
| count grows after rerun | replace append logic with the approved `CREATE OR REPLACE TABLE` pattern |
| lineage is missing | carry `_source_file_name`, `_source_file_path`, `_ingested_at`, `_ingestion_run_id`, `_schema_version` and the record hash through every stage |
| `price_per_sqft_variance` is large for many rows | do not "fix" it here — record it, it is exactly the evidence DQ-04 needs in Week 6 |


## 13. Team ownership and evidence

| Student | Primary responsibility |
|---|---|
| Ms. Thota Madhulika | verify the Week-4 Bronze handoff (Section 3), document the transformation contract (Section 11) |
| Ms. P. Lakshmi Naga Sree | implement the listings Candidate SQL and the six calculated fields (Section 5) |
| Ms. Vadlamuru Rishitha | implement leads/localities/brokers Candidate SQL (Sections 6–8), run reconciliation, lineage and rerun checks (Sections 9–10) |

All three students must review the final notebook and be able to explain at
least one transformation and one validation in the Week-5 viva.

### Required repository evidence

- executed project notebook at `notebooks/03_silver_transformations.ipynb`;
- Candidate schema/sample evidence for all four entities (Sections 5.6, 6, 7, 8);
- count and distinct-key reconciliation evidence (Sections 9.1, 9.4);
- row-level lineage proof for all four entities (Section 9.2);
- rerun proof / Delta history (Section 10);
- updated `weekly/week_05_log.md` and AI Transparency Note.


## 14. Exit checklist

- [ ] I can state the exact Week-5 objective and outputs.
- [ ] All four approved Bronze inputs exist with the full approved column set and remain unchanged.
- [ ] Every approved Candidate table was created as Delta.
- [ ] Only documented standardisations and type conversions were used.
- [ ] All six documented listings calculated fields use typed inputs and approved formulas.
- [ ] No rows were filtered, deduplicated, rejected or quarantined.
- [ ] Bronze count equals Candidate count for every entity (Section 9.1).
- [ ] Bronze distinct `record_uid` count equals Candidate distinct `record_uid` count for every entity (Section 9.4).
- [ ] Every Candidate row traces to a Bronze record hash (Section 9.2).
- [ ] Safe-cast failures remain visible, not silently repaired.
- [ ] Controlled rerun leaves counts stable (Section 10).
- [ ] Notebook, evidence, Week Log and AI Transparency Note are saved in GitHub.
- [ ] No DQ routing, quarantine, Trusted Silver, Gold, Power BI or streaming work is included — that starts in Week 6.


# Stop here — Week 5 complete

```text
Bronze tables (listings, leads, localities, brokers)
        -> standardised views (identifiers + approved domains)
        -> typed views (dates, counts, prices, flags)
        -> calculated views (listings: 6 documented fields)
        -> Silver Candidate Delta tables
        -> reconciliation (row count + distinct record_uid) and row-level lineage proof
        -> rerun proof
        -> GitHub evidence
```

Data quality routing, quarantine, Trusted Silver, Gold tables, Power BI and
streaming belong to later weeks and are out of scope here.
